The purpose is to test whether an agent can learn to seek the goldilocks zone of r1 < r < r2, when the "dum" opponent changes course and speed.
Also to test out how to visualize the different states.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import TimeLimit
import imageio
import os

In [ ]:
class RangeKeepingEnv(gym.Env):
    def __init__(self, agent_speed=1.0, schedule=None):
        super(RangeKeepingEnv, self).__init__()

        # Agent's maximum speed
        self.agent_speed = agent_speed

        # Continuous action space: movement in the x and y directions
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)

        # Continuous state space: agent's position (x, y) and moving point's position (x, y)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(4,), dtype=np.float32)

        # Initial positions of the agent and the moving point
        self.agent_pos = np.array([15.0, 15.0])  # Start the agent at (15, 15)
        self.moving_point_pos = np.array([5.0, 5.0])  # Start the moving point at (5, 5)

        # Desired range from the moving point
        self.desired_range_min = 5
        self.desired_range_max = 10

        # Movement schedule for the moving point
        self.schedule = schedule if schedule is not None else lambda t: np.array([5 + 0.1 * t, 5 + 0.1 * t])

        # Episode step counter
        self.current_step = 0
        self.max_steps = 1000

    def reset(self, seed=None, options=None):
        self.agent_pos = np.random.uniform(-20, 20, size=(2,))
        self.moving_point_pos = self.schedule(0)
        self.current_step = 0
        return np.concatenate([self.agent_pos, self.moving_point_pos]), {}

    def step(self, action):
        self.current_step += 1

        # Reset agent to a random position after every 1000 steps
        if self.current_step % 50 == 0:
            self.agent_pos = np.random.uniform(-20, 20, size=(2,))

        # Calculate the norm of the action vector (x, y)
        action_magnitude = np.linalg.norm(action)

        # If the magnitude of the action exceeds the agent's speed, normalize it
        if action_magnitude > 1.0:
            action = action / action_magnitude  # Normalize to unit vector
    
        # Scale the action to the maximum agent speed
        movement = action * self.agent_speed

        # Update agent's position based on the movement
        self.agent_pos += movement

        # Update the moving point position based on the schedule
        self.moving_point_pos = self.schedule(self.current_step)

        # Calculate the distance between the agent and the moving point
        distance = np.linalg.norm(self.agent_pos - self.moving_point_pos)

        # Reward: Give a positive reward if the agent is within the desired range (5 to 10 units)
        if self.desired_range_min <= distance <= self.desired_range_max:
            reward = 1.0  # Positive reward for being within the desired range
        if distance < self.desired_range_min:
            reward = -100
        else:
            reward = -np.abs(distance - self.desired_range_max)  # Penalize for being outside the range

        # Episode termination
        terminated = self.current_step >= self.max_steps
        truncated = False

        state = np.concatenate([self.agent_pos, self.moving_point_pos])
        return state, reward, terminated, truncated, {}

    def render(self, mode='human'):
        print(f'Step: {self.current_step}, Agent Position: {self.agent_pos}, Moving Point Position: {self.moving_point_pos}')

    def set_agent_speed(self, speed):
        self.agent_speed = speed

    def set_moving_point_schedule(self, schedule):
        self.schedule = schedule

# Create a function to generate the GIF
def create_gif_from_env(env, model,steps, gif_name='range_keeping.gif'):
    frames = []

    fig, ax = plt.subplots()

    # Reset environment
    obs, _ = env.reset()

    for step in range(steps):
        action, _ = model.predict(obs)  # Use the trained model to predict actions
        obs, reward, terminated, truncated, info = env.step(action)

        ax.clear()

        # Plot the moving point and agent
        moving_point = env.moving_point_pos
        agent_point = env.agent_pos

        ax.plot(moving_point[0], moving_point[1], 'ro', label='Opponent')  # Moving point in red
        ax.plot(agent_point[0], agent_point[1], 'bo', label='Agent')  # Agent in blue

        # Draw a circle showing the desired range from the moving point
        circle1 = plt.Circle((moving_point[0], moving_point[1]), env.desired_range_min, color='g', fill=False, linestyle='--',label='Desired Range')
        circle2 = plt.Circle((moving_point[0], moving_point[1]), env.desired_range_max, color='g', fill=False, linestyle='--')
        ax.add_patch(circle1)
        ax.add_patch(circle2)
        ax.set_xlim(-20, 20)
        ax.set_ylim(-20, 20)
        ax.set_title(f'Step {step+1}')
        ax.set_xlabel('X Position')
        ax.set_ylabel('Y Position')
        ax.legend()

        plt.draw()

        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)

        if terminated or truncated:
            break

    imageio.mimsave(gif_name, frames, fps=20)
    print(f"GIF saved as {gif_name}")

class RandomMovementSchedule:
    def __init__(self, speed_limit=1.0):
        self.speed_limit = speed_limit  # Limit speed to half of agent speed
        self.direction = np.random.uniform(-1, 1, size=2)  # Initial random direction
        self.speed = np.random.uniform(0, self.speed_limit)  # Initial random speed
        self.current_step = 0
        self.change_interval = 20  # Change direction and speed every 20 steps
        self.position = np.array([5.0, 5.0])  # Start the moving point at (5, 5)

    def __call__(self, t):
        if t % self.change_interval == 0:
            # Every 20 steps, randomly change speed and direction
            self.direction = np.random.uniform(-1, 1, size=2)
            self.direction /= np.linalg.norm(self.direction)  # Normalize direction vector
            self.speed = np.random.uniform(0, self.speed_limit)

        # Update the position based on the current speed and direction
        self.position += self.direction * self.speed
        self.current_step += 1

        return self.position


def train_agent():
    # Create a simple linear schedule for the moving point
    def simple_schedule(t):
        return np.array([5 + 0.1 * t, 5 + 0.1 * t])  # Linear movement

    def circular_schedule(t, radius=5.0, angular_speed=0.05, center=(0.0, 0.0)):
        x_center, y_center = center
        x = radius * np.cos(angular_speed * t) + x_center
        y = radius * np.sin(angular_speed * t) + y_center
        return np.array([x, y])

    random_schedule = RandomMovementSchedule(agent_speed=1.0)

    # Instantiate the environment
    env = RangeKeepingEnv(agent_speed=1.0, schedule=circular_schedule)

    # Initialize the PPO model with MlpPolicy
    model = PPO('MlpPolicy', env, verbose=1)

    # Train the model for 100,000 steps
    model.learn(total_timesteps=100000)

    # Create a GIF of the trained agent's behavior
    create_gif_from_env(env, model, gif_name="range_keeping_trained.gif", steps=1000)

# Run the training and create the GIF
if __name__ == "__main__":
    train_agent()


NameError: name 'gym' is not defined

In [6]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class RangeKeepingEnv(gym.Env):
    def __init__(self, agent_speed=1.0, schedule=None):
        super(RangeKeepingEnv, self).__init__()

        # Agent's maximum speed
        self.agent_speed = agent_speed

        # Continuous action space: movement in the x and y directions
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)

        # Continuous state space: agent's position (x, y) and moving point's position (x, y)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(4,), dtype=np.float32)

        # Initial positions of the agent and the moving point
        self.agent_pos = np.array([15.0, 15.0])  # Start the agent at (15, 15)
        self.moving_point_pos = np.array([5.0, 5.0])  # Start the moving point at (5, 5)

        # Desired range from the moving point
        self.desired_range_min = 5
        self.desired_range_max = 10

        # Movement schedule for the moving point
        self.schedule = schedule if schedule is not None else lambda t: np.array([5 + 0.1 * t, 5 + 0.1 * t])

        # Episode step counter
        self.current_step = 0
        self.max_steps = 1000

        # Add health (lives) for both the agent and the moving point
        self.agent_lives = 3
        self.moving_point_lives = 3

    def reset(self, seed=None, options=None):
        self.agent_pos = np.random.uniform(-20, 20, size=(2,))
        self.moving_point_pos = self.schedule(0)
        self.current_step = 0
        self.agent_lives = 3  # Reset lives
        self.moving_point_lives = 3  # Reset lives
        return np.concatenate([self.agent_pos, self.moving_point_pos]), {}

    def step(self, action):
        self.current_step += 1

        # Reset agent to a random position after every 50 steps
        if self.current_step % 50 == 0:
            self.agent_pos = np.random.uniform(0, 20, size=(2,))

        # Calculate the norm of the action vector (x, y)
        action_magnitude = np.linalg.norm(action)

        # If the magnitude of the action exceeds the agent's speed, normalize it
        if action_magnitude > 1.0:
            action = action / action_magnitude  # Normalize to unit vector
    
        # Scale the action to the maximum agent speed
        movement = action * self.agent_speed

        # Update agent's position based on the movement
        self.agent_pos += movement

        # Update the moving point position based on the schedule
        self.moving_point_pos = self.schedule(self.current_step)

        # Calculate the distance between the agent and the moving point
        distance = np.linalg.norm(self.agent_pos - self.moving_point_pos)

        # Reward: Give a positive reward if the agent is within the desired range (5 to 10 units)
        if self.desired_range_min <= distance <= self.desired_range_max:
            reward = 1.0  # Positive reward for being within the desired range
        else:
            reward = -np.abs(distance - self.desired_range_max)  # Penalize for being outside the range

        # Shooting mechanic: the agent shoots the moving point if within `desired_range_max`
        if distance <= self.desired_range_max:
            self.moving_point_lives -= 1  # Moving point loses a life
            reward += 10  # Reward for hitting the moving point

        # The moving point shoots the agent if within `desired_range_min`
        if distance <= self.desired_range_min:
            self.agent_lives -= 1  # Agent loses a life
            reward -= 10  # Penalize for being hit

        # Check if either has lost all lives and terminate the episode
        if self.agent_lives <= 0 or self.moving_point_lives <= 0:
            terminated = True
            reward += 100 if self.moving_point_lives <= 0 else -100  # Big reward for agent win, penalty for loss
        else:
            terminated = self.current_step >= self.max_steps

        truncated = False  # No truncation in this scenario

        state = np.concatenate([self.agent_pos, self.moving_point_pos])
        return state, reward, terminated, truncated, {}

    def render(self, mode='human'):
        print(f'Step: {self.current_step}, Agent Position: {self.agent_pos}, Moving Point Position: {self.moving_point_pos}')
        print(f'Agent Lives: {self.agent_lives}, Moving Point Lives: {self.moving_point_lives}')

    def set_agent_speed(self, speed):
        self.agent_speed = speed

    def set_moving_point_schedule(self, schedule):
        self.schedule = schedule
